# Multi-period Charge Mix Planning for Foundry

The "melting planning manager" of a foundry with electric arc furnaces (EAF) decides over a planning horizon of several days to a week for each time block (period) "which order grade of molten metal to melt, how many times (integer), how many tons per time (continuous), and how many tons of which scrap inventory lot to feed into each charge." This refines the existing simpler version (`foundry_charge_mix.py`, single charge, 2 raw materials) to multiple periods, multiple orders, inventory quality variance, and time-of-use electricity pricing.

This problem is affected by **two types of bilinearities** simultaneously:

1. **Number of heats (integer) $\times$ Heat size (continuous) = Melt volume**. Since the electric furnace is a discrete operation requiring setup (power startup, tapping) for each melt (heat), the number of times is an integer, and the charge per time is continuous within the furnace capacity.
2. **Molten metal composition (concentration) $\times$ Melt volume / Allocation volume**. The carbon and copper concentration of a single well-stirred molten metal is determined by the weighted average of the charge mix (concentration $\times$ mass), and since one molten metal is allocated to multiple orders, the orders cast in the same period are coupled through the shared molten metal composition (strong pooling-type non-convexity).

Business meaning of each constraint:

- **Quality variance of scrap inventory**: Recovered scraps have different carbon and copper contents per lot and limited inventory. Since copper cannot be removed by refining, the use of high-copper scrap is restricted by the grade's upper limit (tramp element problem). Inventory is shared across all periods (time coupling).
- **Composition specifications & delivery deadlines**: The allocated orders have carbon specification windows, copper upper limits, and delivery deadlines.
- **Time-of-use electricity pricing**: There are time-of-use rates with cheap nights and expensive days, creating an incentive to consolidate melting during cheap periods.
- **Outsourcing backstop**: Any shortage in meeting delivery deadlines via in-house melting is covered by purchasing spec-compliant outsourced products (always ensuring feasibility).

Subject: `samples/manufacturing_and_blending/foundry_charge_mix_multiperiod.py`.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/manufacturing_and_blending"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display

def show(fig):
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import foundry_charge_mix_multiperiod as fc

C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

## 1. Naive Formulation

The default scale is 12 lots $\times$ 8 orders $\times$ 8 periods. `melt[t] == n[t]*s[t]` (integer $\times$ continuous) and `carbon_bal_t`/`copper_bal_t` (concentration $\times$ mass) are the center of the non-linearity.

In [ ]:
m0 = fc.build_model("default")
nl = sum(1 for c in m0.getConss() if c.isNonlinear())
print(f"変数 {m0.getNVars()}(うち整数 {m0.getNIntVars()})、制約 {m0.getNConss()}"
      f"(うち非線形 {nl})")
kinds = {}
for c in m0.getConss():
    if c.isNonlinear():
        k = c.name.rsplit("_", 1)[0]
        kinds[k] = kinds.get(k, 0) + 1
print("非線形制約の内訳:", kinds)
del m0

`melt_def` (integer $\times$ continuous) appears for the number of periods, while `carbon_bal`/`copper_bal` (concentration $\times$ mass) and `carb_lo`/`carb_hi`/`cu_hi` (composition specs of allocation volume $\times$ concentration) appear for the combinations of orders $\times$ periods.
The latter are overwhelmingly more numerous, creating **pooling-type couplings** that share the molten metal composition `cc[t]`/`cu[t]` among orders.

## 2. Diagnosis

In [ ]:
report = mk.analyze(lambda: fc.build_model("default"), name="foundry-charge-mix",
                    time_limit=30)
print(report.summary())
print("\ngap:", f"{report.metrics.get('gap', 0):.1%}",
      "/ 空間分枝の双対寄与:", f"{report.metrics.get('spatial_share', 0):.1%}",
      "/ 停滞区間数:", report.metrics.get("n_stalls"))

In 30 seconds, a gap of over 50% remains, and `dual_stall` (stagnation of dual bound improvement) triggers. Spatial branches (branches on continuous variables) account for nearly 90% of the dual bound improvement, meaning branches on the discrete side (number of heats) alone do not shrink the gap — suggesting that the pooling-type coupling (sharing of molten metal composition) is the dominant bottleneck. Let's look inside.

## 3. Looking Inside the Diagnosis — Attribution Analysis and Violation Breakdown

We directly probe `mk.collectors.attribution` (which branch variables the dual bound improvement was attributed to) and `mk.collectors.violation` (non-linear constraint violations at the root LP relaxation solution).

In [ ]:
from minlpkit.collectors.attribution import solve_and_attribute, gain_by_kind, gain_by_variable
from minlpkit.collectors.violation import collect_root_violations, violation_by_type

m1 = fc.build_model("default")
d1, summ1 = solve_and_attribute(m1, time_limit=30, gap_limit=0.01)
gk = gain_by_kind(d1)
gk

In [ ]:
fig = go.Figure(layout=base_layout(
    "双対境界の改善の帰属(分枝の種類別、合計に対する割合)",
    "分枝の種類", "双対境界の押し上げ量(合計)", height=380))
fig.add_trace(go.Bar(x=gk["kind"], y=gk["dual_gain"],
    marker_color=[C["warn"] if k == "spatial" else C["s1"] for k in gk["kind"]],
    text=[f"{v:.1f}" for v in gk["dual_gain"]], textposition="outside"))
show(fig)

In [ ]:
m2 = fc.build_model("default")
vdf = collect_root_violations(m2)
vt = violation_by_type(vdf)
vt

In [ ]:
fig = go.Figure(layout=base_layout(
    "非線形制約タイプ別の相対違反(ルートLP緩和解、平均降順)",
    "制約タイプ", "相対違反(平均)", height=380))
fig.add_trace(go.Bar(x=vt["ctype"], y=vt["mean_rel"],
    marker_color=C["warn"], text=[f"{v:.2f}" for v in vt["mean_rel"]],
    textposition="outside"))
fig.update_xaxes(tickangle=-30)
show(fig)

`spatial` (spatial branches on continuous variables) earns almost all of the dual bound improvement, and the contribution of `discrete` (integer branches such as the number of heats) is negligible. In the breakdown of violations, `carbon_bal` (concentration $\times$ mass) is the largest — confirming that the sharing of molten metal composition (pooling-type) is the dominant bottleneck, rather than the integer $\times$ continuous product of heats $\times$ size (`melt_def`). However, since `melt_def` is exactly the form (integer $\times$ continuous) to which `mk.linearize_product` applies directly, we will strictly linearize it next to empirically measure how effective it is.

## 4. Trying Improvements — Strictly Linearize `melt = n・s` with `mk.linearize_product`

`melt[t] == n[t]*s[t]` is a product of the number of heats (integer, 0..4) $\times$ heat size (continuous, 0..30 tons), which is the same integer $\times$ continuous pattern as [Strict Linearization of Integer $\times$ Continuous](../improve/01_linearize_product.ipynb).
We use `mk.linearize_product` to create an exact linear representation, link it with an equality to the existing `melt[t]` to tighten the relaxation (keeping the original bilinear constraint intact so the relaxed domain shrinks to the intersection of both = the side of the strict linear representation).

In [ ]:
def build_linearized(scale="default"):
    '''melt[t] = n[t]*s[t] を mk.linearize_product で厳密線形化した変種。'''
    m = fc.build_model(scale)
    n, s, melt = m.data["n"], m.data["s"], m.data["melt"]
    nI, nO, nT = m.data["dims"]
    for t in range(nT):
        ns = mk.linearize_product(m, n[t], s[t], 0, fc.N_HEAT_MAX, 0.0, fc.HEAT_MAX,
                                  f"ns_{t}")
        m.addCons(melt[t] == ns, name=f"melt_tight_{t}")
    return m

mb = fc.build_model("small"); mb.hideOutput(); mb.optimize()
ml = build_linearized("small"); ml.hideOutput(); ml.optimize()
print(f"baseline   : obj={mb.getObjVal():.2f}  status={mb.getStatus()}")
print(f"linearized : obj={ml.getObjVal():.2f}  status={ml.getStatus()}")

In [ ]:
df = mk.compare_variants(
    {"baseline(bilinear melt)": lambda: fc.build_model("default"),
     "linearized(melt)":        lambda: build_linearized("default")},
    time_limit=30)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

In [ ]:
base = df.loc[df["variant"] == "baseline(bilinear melt)"].iloc[0]
lin  = df.loc[df["variant"] == "linearized(melt)"].iloc[0]
labels = ["baseline", "linearized(melt)"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [base["root_dual"], lin["root_dual"]], lambda v: f"{v:.0f}")
add_bars(2, [base["final_gap"]*100, lin["final_gap"]*100], lambda v: f"{v:.0f}%")
add_bars(3, [base["nodes"], lin["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="melt=n・s の厳密線形化 before / after", x=0.01,
               font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)
print(f"root_dual : {base['root_dual']:.1f} -> {lin['root_dual']:.1f}")
print(f"final_gap : {base['final_gap']:.1%} -> {lin['final_gap']:.1%}")
print(f"nodes     : {int(base['nodes'])} -> {int(lin['nodes'])}")

**Honest verification result**: As shown by the attribution analysis in Section 3, the main cause of the gap in this problem is not `melt=n・s` (integer $\times$ continuous, per period), but the pooling-type coupling of the molten metal composition shared among orders `carbon_bal`/`carb_lo`/`carb_hi` (concentration $\times$ mass, orders $\times$ periods). While the strict linearization of `melt` tightens the relaxation of its corresponding part, it does not touch the dominant bottleneck, so the improvements in `root_dual`/`final_gap`/`nodes` at the default scale can be limited (as measured in the graph above). This result itself substantiates the value of the diagnosis $\rightarrow$ improvement workflow: "Identify the root cause in Section 3 before taking action in Section 4" — simply applying `mk.linearize_product` just by looking at the melt side may fail to reach the root cause (pooling-type molten metal composition coupling).

## Summary

- This problem involves the **coexistence of two types of bilinearities**: (1) Number of heats $\times$ size (integer $\times$ continuous, local), and (2) Molten metal composition $\times$ mass/allocation volume (concentration $\times$ flow rate, shared among orders coupling in a pooling type).
  Attribution and violation analyses provide a way to isolate which is dominant empirically — in this subject, (2) was the main cause.
- Practically, this is the melting plan itself: "Which orders to allocate the limited high-quality scrap (low copper) to, and when to consolidate melting during cheap electricity periods." The sharing of molten metal composition among orders is not an approximation but stems from the operational reality of "making multiple castings from one furnace."
- Lesson: Even if `weak_relaxation`/`dual_stall` triggers, just indiscriminately applying the handy `mk.linearize_product` without identifying "which non-linear term is the root cause" via attribution/violation can yield limited effects.

Related: [Strict Linearization of Integer $\times$ Continuous](../improve/01_linearize_product.ipynb) / [Method Guide 1](../../playbook/01-linearize.md) / API [`mk.linearize_product`](../../api/transforms.md)